In [ ]:
# =========================
# FULL MINI PIPELINE (ONE CELL)
# =========================

import numpy as np
from pyscf import gto, scf, fci

# -------------------------
# Molecule definitions
# -------------------------
def get_water():
    return """
    O  0.000000  0.000000  0.000000
    H  0.758602  0.000000  0.504284
    H -0.758602  0.000000  0.504284
    """

def get_ammonia():
    return """
    N  0.000000  0.000000  0.000000
    H  0.9377    0.0000    0.3816
    H -0.4688    0.8121    0.3816
    H -0.4688   -0.8121    0.3816
    """

def build_complex(distance):
    return f"""
    N   0.000000  0.000000  0.000000
    H   0.9377    0.0000    0.3816
    H  -0.4688    0.8121    0.3816
    H  -0.4688   -0.8121    0.3816
    O   0.000000  0.000000  {distance}
    H   0.758602  0.000000  {distance + 0.504284}
    H  -0.758602  0.000000  {distance + 0.504284}
    """

# -------------------------
# RHF function
# -------------------------
def run_rhf(geometry, basis="sto-3g"):
    mol = gto.Mole()
    mol.atom = geometry
    mol.basis = basis
    mol.verbose = 0
    mol.build()

    mf = scf.RHF(mol)
    energy = mf.kernel()

    return {
        "mol": mol,
        "mf": mf,
        "energy": energy,
        "mo_coeff": mf.mo_coeff,
        "mo_energy": mf.mo_energy,
        "density_matrix": mf.make_rdm1(),
    }

# -------------------------
# Orbital selection (simple)
# -------------------------
def select_active_orbitals(mo_energy, n_active=6):
    n_orb = len(mo_energy)
    center = n_orb // 2
    start = max(center - n_active // 2, 0)
    end = min(center + n_active // 2, n_orb)
    return list(range(start, end))

# -------------------------
# FCI solver
# -------------------------
def run_fci(mol, mf):
    cisolver = fci.FCI(mol, mf.mo_coeff)
    e_fci, _ = cisolver.kernel()
    return e_fci



# -------------------------
# Binding energy
# -------------------------
def compute_binding(distance):
    water = run_rhf(get_water())
    ammonia = run_rhf(get_ammonia())
    complex_sys = run_rhf(build_complex(distance))

    delta_E = (
        complex_sys["energy"]
        - ammonia["energy"]
        - water["energy"]
    )

    return delta_E

# -------------------------
# SINGLE TEST RUN
# -------------------------
print("=== SINGLE RUN ===")
res = run_rhf(get_water())
print("Water RHF Energy:", res["energy"])

active = select_active_orbitals(res["mo_energy"])
print("Active orbitals:", active)

e_fci = run_fci(res["mol"], res["mf"])
print("Water FCI Energy:", e_fci)

# -------------------------
# BINDING ENERGY TEST
# -------------------------
print("\n=== BINDING ENERGY ===")
d = 2.8
delta_E = compute_binding(d)
print(f"Distance = {d} Å")
print("Binding Energy (Hartree):", delta_E)
print("Binding Energy (kcal/mol):", delta_E * 627.5)

# -------------------------
# DISTANCE SCAN
# -------------------------
print("\n=== DISTANCE SCAN ===")
distances = np.linspace(2.0, 5.0, 7)

for d in distances:
    delta_E = compute_binding(d)
    print(f"d = {d:.2f} Å | ΔE = {delta_E:.6f} Ha | {delta_E*627.5:.2f} kcal/mol")

=== SINGLE RUN ===
Water RHF Energy: -74.94588076098955
Active orbitals: [0, 1, 2, 3, 4, 5]
Water FCI Energy: -74.98862945178298

=== BINDING ENERGY ===
Distance = 2.8 Å
Binding Energy (Hartree): -0.00120695803406079
Binding Energy (kcal/mol): -0.7573661663731457

=== DISTANCE SCAN ===
d = 2.00 Å | ΔE = 0.067991 Ha | 42.66 kcal/mol
d = 2.50 Å | ΔE = 0.004673 Ha | 2.93 kcal/mol
d = 3.00 Å | ΔE = -0.002061 Ha | -1.29 kcal/mol
d = 3.50 Å | ΔE = -0.001857 Ha | -1.17 kcal/mol
d = 4.00 Å | ΔE = -0.001366 Ha | -0.86 kcal/mol
d = 4.50 Å | ΔE = -0.001029 Ha | -0.65 kcal/mol
d = 5.00 Å | ΔE = -0.000794 Ha | -0.50 kcal/mol


In [ ]:
# =========================
# SQD SOLVER (QISKIT)
# =========================

from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.problems import ElectronicStructureProblem
from qiskit_nature.second_q.algorithms import GroundStateEigensolver

# SQD solver
from qiskit_addon_sqd import SQDSolver


def run_sqd(geometry, basis="sto-3g"):
    # Convert geometry string → PySCFDriver format
    atom_str = ";".join([line.strip() for line in geometry.strip().split("\n") if line.strip()])

    driver = PySCFDriver(
        atom=atom_str,
        basis=basis,
        charge=0,
        spin=0
    )

    problem = ElectronicStructureProblem(driver)
    second_q_op = problem.second_q_ops()

    # Map fermions → qubits
    mapper = JordanWignerMapper()
    qubit_op = mapper.map(second_q_op[0])

    # SQD solver
    sqd_solver = SQDSolver()

    gsc = GroundStateEigensolver(mapper, sqd_solver)
    result = gsc.solve(problem)

    return result.total_energies[0]

ImportError: cannot import name 'BaseEstimator' from 'qiskit.primitives' (/home/loharkar/QuEnAIS-quantum-embedding/quenais/lib/python3.12/site-packages/qiskit/primitives/__init__.py)